# Open Banking Payment Analysis Homework

Source notebook found in the local auto-network project folder.

This public portfolio copy keeps the full notebook source visible on GitHub while removing execution outputs, execution counts, and environment-specific metadata.

# Final Homework — Payment Analysis for Open Banking

In [ ]:
# ============================================================
# Section 0 — Setup, Load & Initial Inspection
# ============================================================

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

from pathlib import Path

# Display settings for easier notebook reading
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:,.2f}".format)

# File path
DATA_PATH = Path("truelayer_analytics_test_data_set.csv")

# Load data
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Could not find {DATA_PATH}. Make sure the CSV is saved in the same folder as this notebook."
    )

df_raw = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully.")
print("Shape:", df_raw.shape)

display(df_raw.head())
df_raw.info()

In [ ]:
# Overview of columns, data types, missing values, and unique values

column_overview = pd.DataFrame({
    "column": df_raw.columns,
    "dtype": df_raw.dtypes.astype(str).values,
    "non_null_count": df_raw.notna().sum().values,
    "null_count": df_raw.isna().sum().values,
    "null_pct": (df_raw.isna().mean() * 100).round(2).values,
    "unique_values": df_raw.nunique(dropna=True).values
})

display(column_overview)

In [ ]:
# Check availability of fields mentioned or implied by the homework instructions

expected_columns = [
    "id",
    "bank_id",
    "bank_name",
    "bank_type",
    "merchant_id",
    "merchant_plan",
    "customer_id",
    "user_id",
    "vertical",
    "connectivity_type",
    "currency",
    "country_id",
    "status",
    "api_version",
    "failure_reason",
    "failure_stage",
    "amount_in_currency",
    "createdat_ts",
    "lastupdatedat_ts",
    "initiated_at",
    "authorizing_at",
    "authorized_at",
    "executed_at",
    "failed_at",
    "settled_at"
]

column_check = pd.DataFrame({
    "column": expected_columns,
    "present_in_combined_csv": [col in df_raw.columns for col in expected_columns]
})

display(column_check)

missing_expected_columns = column_check.loc[
    ~column_check["present_in_combined_csv"], 
    "column"
].tolist()

print("Missing expected fields:")
print(missing_expected_columns)

The combined dataset contains the core payment-level fields required for most of the analysis: payment IDs, bank IDs, customer IDs, statuses, API versions, transaction amounts, lifecycle timestamps, country, vertical, and connectivity type.

The instructions pdf mention variables such as `merchant_plan`, `bank_type`, and `bank_name` which are not present in CSV, so later sections that request those exact dimensions will be adapted using the closest available fields: `vertical`, `connectivity_type`, `api_version`, `country_id`, and `bank_id`.


In [ ]:
# Create working dataframe

df = df_raw.copy()

print("Working dataframe created.")
print("Shape:", df.shape)

The working dataframe `df` has the same number of rows and columns as the raw dataset. All later transformations will be applied to `df`, while `df_raw` remains unchanged as a reference copy.


The raw `status` column contains mixed casing and several related lifecycle labels. For example, failed payments can appear as `failed`, `Failed`, or `AuthorisationFailed`.

To make later analysis consistent, we create a clean `status_clean` column. So the normalized status will be used for funnel analysis, conversion rates, failure rates, user classification, retention, and growth accounting.

In [ ]:
status_counts = (
    df["status"]
    .value_counts(dropna=False)
    .rename_axis("raw_status")
    .reset_index(name="payment_count")
)

display(status_counts)

In [ ]:
# Create normalized status field

def clean_status(status):
    if pd.isna(status):
        return "unknown"
    
    s = str(status).strip().lower()
    
    status_mapping = {
        "new": "created",
        "submitted": "created",
        "initiated": "initiated",
        "authorizing": "authorizing",
        "authorized": "authorized",
        "executing": "executing",
        "executed": "executed",
        "failed": "failed",
        "authorisationfailed": "failed",
        "cancelled": "cancelled",
        "rejected": "rejected"
    }
    
    return status_mapping.get(s, s)

df["status_clean"] = df["status"].apply(clean_status)

status_clean_summary = (
    df.groupby(["status", "status_clean"], dropna=False)
      .size()
      .reset_index(name="payment_count")
      .sort_values("payment_count", ascending=False)
)

display(status_clean_summary)

The raw `status` field was standardized into `status_clean`.

This reduces inconsistencies caused by mixed casing and related lifecycle labels. So in our case, `failed`, `Failed`, and `AuthorisationFailed` are all treated as failed payments.

I'm using cleaned status field for conversion, funnel, failure, retention, and user lifecycle calculations.

This step converts all lifecycle timestamp columns to datetime format.

The instructions use the term `user_id`, while the dataset contains `customer_id`. so we adapted the customer_id variable for future.

In [ ]:
# Convert timestamp columns to datetime

timestamp_cols = [
    "createdat_ts",
    "lastupdatedat_ts",
    "initiated_at",
    "authorizing_at",
    "authorized_at",
    "executed_at",
    "failed_at",
    "settled_at"
]

for col in timestamp_cols:
    df[col] = pd.to_datetime(df[col], errors="coerce")

# Match instruction wording
df["user_id"] = df["customer_id"]

# Monthly fields for trend, cohort, and growth accounting analysis
df["created_month"] = df["createdat_ts"].dt.to_period("M").dt.to_timestamp()
df["executed_month"] = df["executed_at"].dt.to_period("M").dt.to_timestamp()

# Helper flags for repeated analysis
df["is_executed"] = df["status_clean"].eq("executed")
df["is_failed"] = df["status_clean"].eq("failed")
df["is_cancelled"] = df["status_clean"].eq("cancelled")
df["is_rejected"] = df["status_clean"].eq("rejected")

display(df.head())

In [ ]:
# Section 0 validation summary

section0_summary = pd.DataFrame({
    "metric": [
        "rows",
        "columns_after_preparation",
        "date_min",
        "date_max",
        "unique_payments",
        "unique_users",
        "unique_banks",
        "unique_countries",
        "raw_status_values",
        "clean_status_values"
    ],
    "value": [
        len(df),
        df.shape[1],
        df["createdat_ts"].min(),
        df["createdat_ts"].max(),
        df["id"].nunique(),
        df["user_id"].nunique(),
        df["bank_id"].nunique(),
        df["country_id"].nunique(),
        df["status"].nunique(dropna=False),
        df["status_clean"].nunique(dropna=False)
    ]
})

display(section0_summary)

The dataset contains 500,000 payment records. The original file has 20 columns, and after preparation the working dataframe has 28 columns.

CSV given to us is an already-combined analytical dataset so there is no need for physical join. The main identifiers are `id`, `bank_id`, `customer_id`, `country_id`, and `api_version`. Since the instructions use `user_id`, a `user_id` column was created from `customer_id`.

Our raw `status` column contained 13 different values with mixed casing and related lifecycle labels. These were normalized into 9 clean status categories in `status_clean`, which we will be using for the analysis.

# Section 1 — Data Quality Assessment

Before looking at conversion, volume, or retention, I first check whether the dataset is reliable enough for analysis. This section focuses on missing values, lifecycle timestamp logic, and differences across API versions.

In [ ]:
# Null count and null percentage by column

null_summary = (
    pd.DataFrame({
        "column": df.columns,
        "null_count": df.isna().sum().values,
        "null_pct": (df.isna().mean() * 100).round(2).values,
        "non_null_count": df.notna().sum().values
    })
    .sort_values("null_pct", ascending=False)
    .reset_index(drop=True)
)

display(null_summary)

This table shows how many values are missing in each column. The lifecycle timestamp columns are very important since payments doon't reach the same stage and failed or cancelled payments should not always have `executed_at` or `settled_at`.

In [ ]:
# Focus specifically on lifecycle timestamp columns

lifecycle_cols = [
    "createdat_ts",
    "initiated_at",
    "authorizing_at",
    "authorized_at",
    "executed_at",
    "settled_at",
    "failed_at"
]

lifecycle_null_summary = (
    null_summary[null_summary["column"].isin(lifecycle_cols)]
    .sort_values("null_pct", ascending=False)
    .reset_index(drop=True)
)

display(lifecycle_null_summary)

## Q1.2 — Timestamp order for executed payments

For executed payments, the timestamps should follow the lifecycle order:

`createdat_ts < initiated_at < authorizing_at < authorized_at < executed_at`

HEre we check if this order is broken. These rows may indicate data issues, duplicated timestamps and some inconcistencies.

In [ ]:
# Filter to executed payments

executed_payments = df[df["is_executed"]].copy()

print("Executed payments:", len(executed_payments))

In [ ]:
# Check timestamp ordering rules for executed payments

timestamp_order_checks = {
    "created_before_initiated": executed_payments["createdat_ts"] < executed_payments["initiated_at"],
    "initiated_before_authorizing": executed_payments["initiated_at"] < executed_payments["authorizing_at"],
    "authorizing_before_authorized": executed_payments["authorizing_at"] < executed_payments["authorized_at"],
    "authorized_before_executed": executed_payments["authorized_at"] < executed_payments["executed_at"]
}

timestamp_violations = []

for rule, check in timestamp_order_checks.items():
    valid_comparison = check.fillna(False)
    violations = (~valid_comparison).sum()
    
    timestamp_violations.append({
        "rule": rule,
        "violating_rows": violations,
        "violation_pct_of_executed": round(violations / len(executed_payments) * 100, 2)
    })

timestamp_violations = pd.DataFrame(timestamp_violations)

display(timestamp_violations)

In [ ]:
# Show a small sample of problematic executed payments for inspection

problem_mask = (
    ~(executed_payments["createdat_ts"] < executed_payments["initiated_at"]) |
    ~(executed_payments["initiated_at"] < executed_payments["authorizing_at"]) |
    ~(executed_payments["authorizing_at"] < executed_payments["authorized_at"]) |
    ~(executed_payments["authorized_at"] < executed_payments["executed_at"])
)

timestamp_problem_sample = executed_payments.loc[
    problem_mask,
    [
        "id",
        "status",
        "status_clean",
        "createdat_ts",
        "initiated_at",
        "authorizing_at",
        "authorized_at",
        "executed_at"
    ]
].head(10)

display(timestamp_problem_sample)

This check identifies executed payments where at least one lifecycle timestamp does not follow the expected order.  it doesnt mean that the payments are wrong but that they might beenrecorded at same time or that some lifecycle fields have been inputed differently.

## Q1.3 — Data quality by API version

Next, I compare the main payment outcomes by `api_version`. This helps show whether one version has systematically different success rates, failure rates, or missing failure reasons.


In [ ]:
# API version quality summary

api_version_summary = (
    df.groupby("api_version")
      .agg(
          payment_count=("id", "count"),
          executed_count=("is_executed", "sum"),
          failed_count=("is_failed", "sum")
      )
      .reset_index()
)

api_version_summary["executed_pct"] = (
    api_version_summary["executed_count"] / api_version_summary["payment_count"] * 100
).round(2)

api_version_summary["failed_pct"] = (
    api_version_summary["failed_count"] / api_version_summary["payment_count"] * 100
).round(2)

display(api_version_summary)

In [ ]:
# Among failed payments, check missing failure_reason by API version

failed_reason_summary = (
    df[df["is_failed"]]
    .groupby("api_version")
    .agg(
        failed_payment_count=("id", "count"),
        failed_with_null_reason=("failure_reason", lambda x: x.isna().sum())
    )
    .reset_index()
)

failed_reason_summary["failed_null_reason_pct"] = (
    failed_reason_summary["failed_with_null_reason"] /
    failed_reason_summary["failed_payment_count"] * 100
).round(2)

api_version_quality = api_version_summary.merge(
    failed_reason_summary,
    on="api_version",
    how="left"
)

display(api_version_quality)

There are clear differences between API versions.

`v3` has the largest number of payments and the highest execution rate at 79.44%, but it has a higher failure rate at 18.50% whereas, `v1` and `v2` have much lower failure rates of 0.50% and 0.13%.

The biggest data quality difference is in `failure_reason`. For `v1` and `v2`, 100% of failed payments have a missing failure reason. For `v3`, none of the failed payments have a missing failure reason.
 It s possible that earlier API versions didnt record failure reasons in the same way, and `v3` is a newer approach with failure tracking. Because of this, failure reason analysis will be interpreted mainly using `v3`, or at least with a warning that `v1` and `v2` have incomplete failure reason data.

In [ ]:
# Q1.4 — Bank ID completeness in the combined dataset

bank_id_quality = pd.DataFrame({
    "metric": [
        "total_payments",
        "payments_with_null_bank_id",
        "payments_with_null_bank_id_pct",
        "unique_bank_ids"
    ],
    "value": [
        len(df),
        df["bank_id"].isna().sum(),
        round(df["bank_id"].isna().mean() * 100, 2),
        df["bank_id"].nunique()
    ]
})

display(bank_id_quality)

In [ ]:
# Payment count by bank_id, useful for later bank-level analysis

bank_payment_counts = (
    df.groupby("bank_id")
      .agg(payment_count=("id", "count"))
      .reset_index()
      .sort_values("payment_count", ascending=False)
)

display(bank_payment_counts.head(10))

In [ ]:
# Section 1 compact quality summary

section1_summary = pd.DataFrame({
    "check": [
        "Columns with any null values",
        "Executed payments checked for timestamp order",
        "API versions compared",
        "Bank IDs available",
        "Null bank_id percentage"
    ],
    "result": [
        (df.isna().sum() > 0).sum(),
        len(executed_payments),
        df["api_version"].nunique(),
        df["bank_id"].nunique(),
        round(df["bank_id"].isna().mean() * 100, 2)
    ]
})

display(section1_summary)

The data quality checks show that the dataset is usable ,t here are 11 columns with some missing values, but itss expected since payment lifecycle fields only populate when a payment reaches that stage. For example, failed payments should have failure-related fields, while executed or settled payments should not always have them.

For the bank check, all payments have a `bank_id`, and in this dataset there are 203 unique banks. but in our case the original `banks.csv` reference table is not available separately so I cannot directly calculate the percentage of bank IDs missing from `banks.csv`. Also there are no null `bank_id` values.


In [ ]:
# ============================================================
# Section 2 — Funnel & Conversion Analysis
# ============================================================

# 2.1  Build funnel from lifecycle timestamps

df['is_settled'] = df['settled_at'].notna()

total       = len(df)
initiated   = df['initiated_at'].notna().sum()
authorizing = df['authorizing_at'].notna().sum()
authorized  = df['authorized_at'].notna().sum()
executed    = df['is_executed'].sum()
settled     = df['is_settled'].sum()

funnel = pd.DataFrame({
    'Stage': ['Created','Initiated','Authorizing','Authorized','Executed','Settled'],
    'Count': [total, initiated, authorizing, authorized, executed, settled]
})

funnel['drop_off_pct'] = (
    (funnel['Count'].shift(1) - funnel['Count']) / funnel['Count'].shift(1) * 100
).fillna(0).round(2)

funnel['cumulative_pct'] = (funnel['Count'] / total * 100).round(2)

print(funnel.to_string(index=False))

In [ ]:
# 2.2  Funnel chart

fig, ax = plt.subplots(figsize=(10, 5))

bars = ax.barh(
    funnel['Stage'][::-1],
    funnel['Count'][::-1],
    alpha=0.85
)

for bar, row in zip(bars, funnel[::-1].itertuples()):
    ax.text(
        bar.get_width() + 200,
        bar.get_y() + bar.get_height() / 2,
        f'{row.Count:,} ({row.cumulative_pct:.1f}%)',
        va='center',
        fontsize=9
    )

ax.set_title('Payment Funnel — Volume by Stage')
ax.set_xlabel('Number of Payments')
plt.tight_layout()
plt.show()

#### Q2.1 Answer

The funnel starts with 500,000 payments. Here 47.5% have an `initiated_at` timestamp, and 47.9% have an `authorizing_at` timestamp. The authorized stage is lower at 39.6% of created payments.

The executed stage is higher than the authorized stage = 377,616 executed payments and 75.5% of all created payments. This means the lifecycle timestamps are not perfectly sequential in the dataset: many payments are marked as executed even when earlier authorization timestamps are missing.

Only 73,360 payments, or 14.7% of created payments have a settlement timestamp. Which could mean that settlement is tracked separately from execution, and not every executed payment has a recorded `settled_at` value in the dataset.

In [ ]:
# 2.2  E2E conversion rate by key dimensions

def conversion_by_dimension(data, dim):
    summary = (
        data.groupby(dim, dropna=False)
            .agg(
                payments=('id', 'count'),
                executed=('is_executed', 'sum')
            )
            .reset_index()
    )
    
    summary['conversion_pct'] = (
        summary['executed'] / summary['payments'] * 100
    ).round(2)
    
    return summary.sort_values('conversion_pct', ascending=False)


dimensions = [
    'vertical',
    'merchant_plan',
    'connectivity_type',
    'bank_type',
    'api_version'
]

available_dims = [d for d in dimensions if d in df.columns]
missing_dims = [d for d in dimensions if d not in df.columns]

print("Available dimensions:", available_dims)
print("Missing dimensions:", missing_dims)

In [ ]:
# Conversion by vertical

vertical_conversion = conversion_by_dimension(df, 'vertical')

print(vertical_conversion.to_string(index=False))

In [ ]:
# Conversion by connectivity type

connectivity_conversion = conversion_by_dimension(df, 'connectivity_type')

print(connectivity_conversion.to_string(index=False))

In [ ]:
# Conversion by API version

api_conversion = conversion_by_dimension(df, 'api_version')

print(api_conversion.to_string(index=False))

#### Q2.2 Answer

conversion breakdowns are calculated for `vertical`, `connectivity_type`, and `api_version`. the combined dataset does not include `merchant_plan` or `bank_type`, so I omitted them/

Conversion differs noticeably by segment. By vertical, `vertical 5` has the highest conversion rate at 80.75%, followed by `vertical 4` at 79.58%. `vertical 3` and the rows with missing vertical values have much lower conversion, although their payment counts are small.

By connectivity type, `type 8` has the strongest conversion rate at 78.07% and also the largest payment volume. Missing connectivity type has the weakest conversion rate at 36.50%, which could mean  missing connectivity information is due to weaker payment performance or incomplete records.

By API version, conversion improves from `v1` to `v3`: `v1` converts at 71.08%, `v2` at 72.35%, and `v3` at 79.44%.  `v3` performs better on end-to-end execution, but when checking earlier data quality, it showed that `v3` has a much higher recorded failure rate.

In [ ]:
# 2.3  Monthly E2E conversion with 3-month rolling average

monthly = (
    df.groupby('created_month')
      .agg(
          payments=('id', 'count'),
          executed=('is_executed', 'sum')
      )
      .reset_index()
)

monthly['conversion_pct'] = (
    monthly['executed'] / monthly['payments'] * 100
).round(2)

monthly['conversion_3m_avg'] = (
    monthly['conversion_pct']
    .rolling(window=3, min_periods=1)
    .mean()
    .round(2)
)

print(monthly.head().to_string(index=False))
print(monthly.tail().to_string(index=False))

In [ ]:
# Plot monthly conversion and 3-month rolling average

fig, ax = plt.subplots(figsize=(12, 5))

ax.plot(
    monthly['created_month'],
    monthly['conversion_pct'],
    marker='o',
    linewidth=1,
    label='Monthly conversion'
)

ax.plot(
    monthly['created_month'],
    monthly['conversion_3m_avg'],
    linewidth=2,
    label='3-month rolling average'
)

ax.set_title('Monthly E2E Conversion Rate')
ax.set_xlabel('Created Month')
ax.set_ylabel('Conversion Rate (%)')
ax.legend()

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

#### Q2.3 Answer

Monthly E2E conversion improves over time.

The earliest months have very small payment volumes, so their conversion rates are unstable and not very reliable. After the dataset reaches meaningful monthly volume, conversion becomes much more stable.

From around 2006 to 2008, conversion is mostly in the mid-60% to low-70% range. After 2008, the trend improve reaching around 80% by the beginning of 2010. The 3-month rolling average confirms the same pattern, so overall conversion appears to be improving rather than declining or staying flat.

In [ ]:
# 2.4  Monthly E2E conversion by country

country_monthly = (
    df.groupby(['created_month', 'country_id'])
      .agg(
          payments=('id', 'count'),
          executed=('is_executed', 'sum')
      )
      .reset_index()
)

country_monthly['conversion_pct'] = (
    country_monthly['executed'] / country_monthly['payments'] * 100
).round(2)

# Keep only country-months with enough volume to avoid noisy tiny samples
country_monthly_filtered = country_monthly[country_monthly['payments'] >= 100].copy()

country_pivot = country_monthly_filtered.pivot(
    index='country_id',
    columns='created_month',
    values='conversion_pct'
)

display(country_monthly_filtered.head())
display(country_pivot)

In [ ]:
# Heatmap using matplotlib

fig, ax = plt.subplots(figsize=(14, 6))

im = ax.imshow(
    country_pivot,
    aspect='auto'
)

ax.set_title('Monthly E2E Conversion by Country')
ax.set_xlabel('Created Month')
ax.set_ylabel('Country')

ax.set_yticks(range(len(country_pivot.index)))
ax.set_yticklabels(country_pivot.index)

x_labels = [str(x)[:7] for x in country_pivot.columns]
ax.set_xticks(range(len(x_labels)))
ax.set_xticklabels(x_labels, rotation=90)

cbar = plt.colorbar(im, ax=ax)
cbar.set_label('Conversion Rate (%)')

plt.tight_layout()
plt.show()

In [ ]:
# Identify sudden country-level conversion drops month over month

country_monthly_filtered = country_monthly_filtered.sort_values(
    ['country_id', 'created_month']
)

country_monthly_filtered['prev_conversion_pct'] = (
    country_monthly_filtered
    .groupby('country_id')['conversion_pct']
    .shift(1)
)

country_monthly_filtered['conversion_change_pp'] = (
    country_monthly_filtered['conversion_pct'] -
    country_monthly_filtered['prev_conversion_pct']
).round(2)

country_drops = (
    country_monthly_filtered[
        country_monthly_filtered['conversion_change_pp'] <= -10
    ]
    .sort_values('conversion_change_pp')
)

print(country_drops.head(15).to_string(index=False))

In [ ]:
# Cross-reference country drops with API version mix in the same month

if len(country_drops) > 0:
    drop_months = country_drops[['country_id', 'created_month']].drop_duplicates()

    drop_api_mix = df.merge(
        drop_months,
        on=['country_id', 'created_month'],
        how='inner'
    )

    drop_api_summary = (
        drop_api_mix
        .groupby(['country_id', 'created_month', 'api_version'])
        .agg(
            payments=('id', 'count'),
            executed=('is_executed', 'sum')
        )
        .reset_index()
    )

    drop_api_summary['conversion_pct'] = (
        drop_api_summary['executed'] / drop_api_summary['payments'] * 100
    ).round(2)

    print(drop_api_summary.sort_values(
        ['country_id', 'created_month', 'api_version']
    ).to_string(index=False))
else:
    print("No country-month conversion drops of 10 percentage points or more were found.")

#### Q2.4 Answer

The country-month heatmap shows that conversion is considerably different across countries. One country has payment activity across almost the full period and shows a clear improvement over time, reaching conversion rates over 80% by the end.

Several country-months show sudden conversion drops of more than 10 percentage points. The largest drops include country `229f...5396` in 2009-03 and 2009-05, both under API version `v2`, and country `2709...dc57` in 2010-01 under API version `v3`.

These drops do not appear to come from one single API version only. Instead, the drop table shows examples under `v1`, `v2`, and `v3`. This suggests that country-level conversion changes may be driven by local market, bank, or volume mix effects than only by API version.

In [ ]:
# ============================================================
# Section 3 — Volume, TPV & AOV
# ============================================================

# 3.1  Overall payment volume metrics

executed_df = df[df['is_executed']].copy()
settled_df = df[df['is_settled']].copy()

tpv = executed_df['amount_in_currency'].sum()
executed_txn_count = len(executed_df)
settled_payment_count = len(settled_df)

aov = (
    settled_df['amount_in_currency'].sum() / settled_payment_count
    if settled_payment_count > 0 else np.nan
)

unique_users = executed_df['user_id'].nunique()
payments_per_user = executed_txn_count / unique_users

volume_summary = pd.DataFrame({
    'Metric': [
        'TPV',
        'Executed transaction count',
        'Settled payment count',
        'AOV',
        'Unique users',
        'Payments per user'
    ],
    'Value': [
        round(tpv, 2),
        executed_txn_count,
        settled_payment_count,
        round(aov, 2),
        unique_users,
        round(payments_per_user, 2)
    ]
})

print(volume_summary.to_string(index=False))

In [ ]:
# 3.2  Summary table by vertical and connectivity type
# merchant_plan is not available in the combined CSV, so connectivity_type is used instead.

vertical_connectivity = (
    executed_df
    .groupby(['vertical', 'connectivity_type'], dropna=False)
    .agg(
        total_tpv=('amount_in_currency', 'sum'),
        transaction_count=('id', 'count')
    )
    .reset_index()
)

vertical_connectivity['aov'] = (
    vertical_connectivity['total_tpv'] / vertical_connectivity['transaction_count']
).round(2)

vertical_connectivity['pct_of_total_tpv'] = (
    vertical_connectivity['total_tpv'] / vertical_connectivity['total_tpv'].sum() * 100
).round(2)

vertical_connectivity = vertical_connectivity.sort_values(
    'total_tpv',
    ascending=False
)

print(vertical_connectivity.head(15).to_string(index=False))

In [ ]:
# Top volume combination

top_combo = vertical_connectivity.head(1)

print(top_combo.to_string(index=False))

In [ ]:
# 3.3  Monthly TPV and AOV

monthly_executed = (
    executed_df
    .groupby('created_month')
    .agg(
        tpv=('amount_in_currency', 'sum'),
        executed_transactions=('id', 'count')
    )
    .reset_index()
)

monthly_settled = (
    settled_df
    .groupby('created_month')
    .agg(
        settled_amount=('amount_in_currency', 'sum'),
        settled_payments=('id', 'count')
    )
    .reset_index()
)

monthly_volume = monthly_executed.merge(
    monthly_settled,
    on='created_month',
    how='left'
)

monthly_volume['aov'] = (
    monthly_volume['settled_amount'] / monthly_volume['settled_payments']
).round(2)

monthly_volume['tpv'] = monthly_volume['tpv'].round(2)

print(monthly_volume.head().to_string(index=False))
print(monthly_volume.tail().to_string(index=False))

In [ ]:
# Plot monthly TPV as bars and monthly AOV as line

fig, ax1 = plt.subplots(figsize=(12, 5))

ax1.bar(
    monthly_volume['created_month'],
    monthly_volume['tpv'],
    width=20,
    alpha=0.75
)

ax1.set_title('Monthly TPV and AOV')
ax1.set_xlabel('Created Month')
ax1.set_ylabel('TPV')

ax2 = ax1.twinx()

ax2.plot(
    monthly_volume['created_month'],
    monthly_volume['aov'],
    marker='o',
    linewidth=2
)

ax2.set_ylabel('AOV')

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

#### Section 3 Answer

Total executed TPV is 10.34 million across 377,616 executed transactions. The dataset has 73,360 settled payments, and the settled-payment AOV is 13.55.

The volume is highly concentrated. Because `merchant_plan` isnt present in the csv, I used `vertical × connectivity_type` as the closest available breakdown. The largest combination is `vertical 2` with `type 8`, which contributes 8.70 million TPV, 233,899 transactions, and 84.09% of total TPV. This is clearly the main driver of volume.

Monthly TPV grows strongly over time, especially from 2008 and further. AOV does not move in the same direction as TPV. In the later period, TPV increases while AOV is mostly flat or declining, which could mean that growth is mainly coming from more transactions rather than larger average payment size.
The early monthly AOV values should be interpreted carefully because several early months have no settled payments, so AOV cannot be calculated for those months.

Q4.1 failure rate by reason and stage

In [ ]:
# ============================================================
# Section 4 — Failure Analysis
# ============================================================

# 4.1  Overall failure rate

failed_df = df[df['is_failed']].copy()

total_payments = len(df)
failed_payments = len(failed_df)

failure_rate = failed_payments / total_payments * 100

print(f"Total payments: {total_payments:,}")
print(f"Failed payments: {failed_payments:,}")
print(f"Overall failure rate: {failure_rate:.2f}%")

In [ ]:
# Failure breakdown by reason

failure_reason_summary = (
    failed_df
    .groupby('failure_reason', dropna=False)
    .agg(
        failed_payments=('id', 'count')
    )
    .reset_index()
)

failure_reason_summary['failure_share_pct'] = (
    failure_reason_summary['failed_payments'] / failed_payments * 100
).round(2)

failure_reason_summary = failure_reason_summary.sort_values(
    'failed_payments',
    ascending=False
)

print(failure_reason_summary.to_string(index=False))

In [ ]:
# Failure breakdown by stage

failure_stage_summary = (
    failed_df
    .groupby('failure_stage', dropna=False)
    .agg(
        failed_payments=('id', 'count')
    )
    .reset_index()
)

failure_stage_summary['failure_share_pct'] = (
    failure_stage_summary['failed_payments'] / failed_payments * 100
).round(2)

failure_stage_summary = failure_stage_summary.sort_values(
    'failed_payments',
    ascending=False
)

print(failure_stage_summary.to_string(index=False))

In [ ]:
# Failure reason by failure stage

failure_reason_stage = (
    failed_df
    .groupby(['failure_reason', 'failure_stage'], dropna=False)
    .agg(
        failed_payments=('id', 'count')
    )
    .reset_index()
)

failure_reason_stage['failure_share_pct'] = (
    failure_reason_stage['failed_payments'] / failed_payments * 100
).round(2)

failure_reason_stage = failure_reason_stage.sort_values(
    'failed_payments',
    ascending=False
)

print(failure_reason_stage.head(20).to_string(index=False))

#### Q4.1 Answer

The overall failure rate is 9.06%, with 45,305 failed payments out of 500,000 total payments.

The most common failure reason is `expired`, which accounts for 23,173 failed payments, or 51.15% of all failures. The next largest reasons are `not_authorized` and `provider_rejected`, but both are much smaller than `expired`.

Failures are concentrated mainly at the `initiated` stage, which accounts for 51.45% of failed payments. The largest reason-stage combination is `expired` at the `initiated` stage, with 13,240 failures. This means that many failures happen early in the payment journey before the user or provider completes the next step.

In [ ]:
# 4.2  Bank failure / cancellation / success rates

bank_summary = df.groupby('bank_id').agg(
    total=('id', 'count'),
    failed=('is_failed', 'sum'),
    cancelled=('is_cancelled', 'sum'),
    executed=('is_executed', 'sum')
)

bank_summary = bank_summary[bank_summary['total'] >= 30].copy()

bank_summary['failure_rate'] = (bank_summary['failed'] / bank_summary['total'] * 100).round(2)
bank_summary['cancellation_rate'] = (bank_summary['cancelled'] / bank_summary['total'] * 100).round(2)
bank_summary['success_rate'] = (bank_summary['executed'] / bank_summary['total'] * 100).round(2)

bank_summary['bank_short'] = bank_summary.index.str[:14] + '...'

failure_threshold = bank_summary['failure_rate'].mean() + 2 * bank_summary['failure_rate'].std()
cancel_threshold = bank_summary['cancellation_rate'].mean() + 2 * bank_summary['cancellation_rate'].std()

bank_summary['outlier'] = (
    (bank_summary['failure_rate'] > failure_threshold) |
    (bank_summary['cancellation_rate'] > cancel_threshold)
)

print('Top 10 Banks by Failure Rate (min 30 payments):')
print(
    bank_summary
    .sort_values('failure_rate', ascending=False)
    .head(10)[['bank_short','total','failed','cancelled','executed',
               'failure_rate','cancellation_rate','success_rate','outlier']]
    .to_string()
)

print()
print(f'Failure outlier threshold     : {failure_threshold:.2f}%')
print(f'Cancellation outlier threshold: {cancel_threshold:.2f}%')
print(f'Outlier banks                 : {bank_summary["outlier"].sum()}')

In [ ]:
# Outlier bank table

outlier_banks = (
    bank_summary[bank_summary['outlier']]
    .sort_values(['failure_rate', 'cancellation_rate'], ascending=False)
)

print('Outlier Banks:')
print(
    outlier_banks[['bank_short','total','failed','cancelled','executed',
                   'failure_rate','cancellation_rate','success_rate']]
    .to_string()
)

In [ ]:
# bank_type is not available, so outliers are highlighted instead.

fig, ax = plt.subplots(figsize=(10, 5))

normal = bank_summary[~bank_summary['outlier']]
outliers = bank_summary[bank_summary['outlier']]

ax.scatter(
    normal['failure_rate'],
    normal['cancellation_rate'],
    s=(normal['total'] / bank_summary['total'].max()) * 500,
    alpha=0.45,
    label='Other banks'
)

ax.scatter(
    outliers['failure_rate'],
    outliers['cancellation_rate'],
    s=(outliers['total'] / bank_summary['total'].max()) * 500,
    alpha=0.8,
    label='Outlier banks'
)

ax.axvline(failure_threshold, linestyle='--', linewidth=1)
ax.axhline(cancel_threshold, linestyle='--', linewidth=1)

ax.set_title('Bank Failure Rate vs Cancellation Rate')
ax.set_xlabel('Failure Rate (%)')
ax.set_ylabel('Cancellation Rate (%)')
ax.legend()

plt.tight_layout()
plt.show()

The bank-level analysis keeps only banks with at least 30 payments. For each bank, I calculated failure rate, cancellation rate, and success rate.

The failure outlier threshold is 35.31%, and the cancellation outlier threshold is 33.34%. In total, 16 banks are flagged as outliers. Some banks are failure-rate outliers, while others are cancellation-rate outliers.

The highest failure-rate bank has 305 payments, 264 failures, and a failure rate of 86.56%. Several other banks also have failure rates above the threshold, including banks with failure rates of 48.18%, 45.76%, and 39.29%.

The original instruction asks for the scatter plot to be coloured by `bank_type`, but this field is not available in the combined CSV. Because of that, the plot highlights outlier banks instead. Most banks are clustered near low failure and low cancellation rates, while a smaller group of banks sits far away from the main cluster and should be investigated further.

In [ ]:
# 4.3  Focus on expired failures

expired_df = failed_df[failed_df['failure_reason'] == 'expired'].copy()

expired_failures = len(expired_df)
expired_share = expired_failures / failed_payments * 100

print(f"Expired failures: {expired_failures:,}")
print(f"Share of all failures: {expired_share:.2f}%")

In [ ]:
# Expired failure rate by API version

expired_api = (
    df.groupby('api_version')
      .agg(
          payments=('id', 'count'),
          failures=('is_failed', 'sum'),
          expired_failures=('failure_reason', lambda x: (x == 'expired').sum())
      )
      .reset_index()
)

expired_api['failure_rate'] = (
    expired_api['failures'] / expired_api['payments'] * 100
).round(2)

expired_api['expired_rate_all_payments'] = (
    expired_api['expired_failures'] / expired_api['payments'] * 100
).round(2)

expired_api['expired_share_of_failures'] = (
    expired_api['expired_failures'] / expired_api['failures'] * 100
).round(2)

print(expired_api.to_string(index=False))

In [ ]:
# Expired failure concentration by bank
# bank_type is not available, so bank_id is used as the closest available bank-level cut.

expired_bank = (
    df.groupby('bank_id')
      .agg(
          payments=('id', 'count'),
          failures=('is_failed', 'sum'),
          expired_failures=('failure_reason', lambda x: (x == 'expired').sum())
      )
)

expired_bank = expired_bank[expired_bank['payments'] >= 30].copy()

expired_bank['failure_rate'] = (
    expired_bank['failures'] / expired_bank['payments'] * 100
).round(2)

expired_bank['expired_rate_all_payments'] = (
    expired_bank['expired_failures'] / expired_bank['payments'] * 100
).round(2)

expired_bank['expired_share_of_failures'] = (
    expired_bank['expired_failures'] / expired_bank['failures'] * 100
).round(2)

expired_bank['bank_short'] = expired_bank.index.str[:14] + '...'

print(
    expired_bank
    .sort_values('expired_rate_all_payments', ascending=False)
    .head(15)[['bank_short','payments','failures','expired_failures',
               'failure_rate','expired_rate_all_payments','expired_share_of_failures']]
    .to_string()
)

In [ ]:
# Plot expired failure rate by API version

fig, ax = plt.subplots(figsize=(8, 4))

ax.bar(
    expired_api['api_version'],
    expired_api['expired_rate_all_payments']
)

ax.set_title('Expired Failure Rate by API Version')
ax.set_xlabel('API Version')
ax.set_ylabel('Expired Failures as % of All Payments')

plt.tight_layout()
plt.show()

`expired` is the dominant failure reason. There are 23,173 expired failures, which represents 51.15% of all failed payments.

The expired failure pattern differs strongly by API version. `v1` and `v2` have zero recorded expired failures, while `v3` has all 23,173 expired failures. For `v3`, expired failures make up 9.61% of all payments and 51.93% of failed payments.

This does not necessarily mean that expiry only existed in `v3`. Earlier data quality checks showed that failed payments in `v1` and `v2` have missing `failure_reason`, so the difference is likely partly caused by better failure reason tracking in `v3`.

The instruction also asks for comparison by `bank_type`, but `bank_type` is not available in the combined CSV. As a substitute, I checked expiry concentration by `bank_id`. Some banks have very high expired failure rates, including one bank where 72.13% of all payments are expired failures. These banks should be investigated because they may have timeout, user journey, or provider-side completion issues.

In [ ]:
# ============================================================
# Section 5 — Latency Analysis
# ============================================================

# 5.1  Latency between consecutive payment stages

latency_df = executed_df.copy()

latency_df['created_to_initiated_sec'] = (
    latency_df['initiated_at'] - latency_df['createdat_ts']
).dt.total_seconds()

latency_df['initiated_to_authorizing_sec'] = (
    latency_df['authorizing_at'] - latency_df['initiated_at']
).dt.total_seconds()

latency_df['authorizing_to_authorized_sec'] = (
    latency_df['authorized_at'] - latency_df['authorizing_at']
).dt.total_seconds()

latency_df['authorized_to_executed_sec'] = (
    latency_df['executed_at'] - latency_df['authorized_at']
).dt.total_seconds()

latency_df['created_to_executed_sec'] = (
    latency_df['executed_at'] - latency_df['createdat_ts']
).dt.total_seconds()

latency_cols = [
    'created_to_initiated_sec',
    'initiated_to_authorizing_sec',
    'authorizing_to_authorized_sec',
    'authorized_to_executed_sec',
    'created_to_executed_sec'
]

latency_summary = []

for col in latency_cols:
    x = latency_df[col].dropna()
    x = x[x >= 0]
    
    latency_summary.append({
        'latency_metric': col,
        'rows': len(x),
        'mean': round(x.mean(), 2),
        'median': round(x.median(), 2),
        'p95': round(x.quantile(0.95), 2),
        'p99': round(x.quantile(0.99), 2)
    })

latency_summary = pd.DataFrame(latency_summary)

print(latency_summary.to_string(index=False))

In [ ]:
# 5.2  E2E latency histogram capped at 600 seconds

e2e_latency = latency_df['created_to_executed_sec'].dropna()
e2e_latency = e2e_latency[e2e_latency >= 0]

e2e_latency_capped = e2e_latency[e2e_latency <= 600]

median_latency = e2e_latency.median()
p95_latency = e2e_latency.quantile(0.95)

print(f"Median E2E latency: {median_latency:.2f} seconds")
print(f"P95 E2E latency   : {p95_latency:.2f} seconds")
print(f"Rows in histogram : {len(e2e_latency_capped):,}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

ax.hist(
    e2e_latency_capped,
    bins=50,
    alpha=0.75
)

ax.axvline(median_latency, linestyle='--', linewidth=2, label='Median')
ax.axvline(p95_latency, linestyle='--', linewidth=2, label='P95')

ax.set_title('E2E Latency Distribution (Capped at 600s)')
ax.set_xlabel('Created to Executed Latency (seconds)')
ax.set_ylabel('Payment Count')
ax.legend()

plt.tight_layout()
plt.show()

#### Q5.1–Q5.2 Answer

For executed payments, the median end-to-end latency from creation to execution is 34.69 seconds, while the p95 is 112.14 seconds. This means most payments complete within roughly one to two minutes.

The distribution is clearly right-skewed. Most payments are concentrated at low latency values, but a smaller number of payments take much longer. This is also visible from the gap between the mean and the median where mean E2E latency is higher than the median because of long-tail cases.

A long tail in payment latency can hurt user experience because users may abandon the flow, retry the payment, or assume the payment failed even if it eventually completes.

In [ ]:
# 5.3  E2E latency by connectivity type and bank

latency_df['e2e_latency_sec'] = latency_df['created_to_executed_sec']

latency_valid = latency_df[
    latency_df['e2e_latency_sec'].notna() &
    (latency_df['e2e_latency_sec'] >= 0)
].copy()

connectivity_latency = (
    latency_valid
    .groupby('connectivity_type', dropna=False)
    .agg(
        payments=('id', 'count'),
        median_latency=('e2e_latency_sec', 'median'),
        p95_latency=('e2e_latency_sec', lambda x: x.quantile(0.95))
    )
    .reset_index()
)

connectivity_latency['median_latency'] = connectivity_latency['median_latency'].round(2)
connectivity_latency['p95_latency'] = connectivity_latency['p95_latency'].round(2)

connectivity_latency = connectivity_latency.sort_values('p95_latency', ascending=False)

print(connectivity_latency.to_string(index=False))

In [ ]:
# Bar chart by connectivity type

fig, ax = plt.subplots(figsize=(9, 5))

ax.bar(
    connectivity_latency['connectivity_type'].astype(str),
    connectivity_latency['p95_latency']
)

ax.set_title('P95 E2E Latency by Connectivity Type')
ax.set_xlabel('Connectivity Type')
ax.set_ylabel('P95 Latency (seconds)')

plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
# Bank-level latency table sorted by p95
# Keep banks with enough executed payments to make p95 meaningful.

bank_latency = (
    latency_valid
    .groupby('bank_id')
    .agg(
        payments=('id', 'count'),
        median_latency=('e2e_latency_sec', 'median'),
        p95_latency=('e2e_latency_sec', lambda x: x.quantile(0.95))
    )
)

bank_latency = bank_latency[bank_latency['payments'] >= 30].copy()

bank_latency['median_latency'] = bank_latency['median_latency'].round(2)
bank_latency['p95_latency'] = bank_latency['p95_latency'].round(2)
bank_latency['bank_short'] = bank_latency.index.str[:14] + '...'

bank_latency = bank_latency.sort_values('p95_latency', ascending=False)

print(
    bank_latency
    .head(15)[['bank_short','payments','median_latency','p95_latency']]
    .to_string()
)

In [ ]:
# Check whether high latency is correlated with high failure rate

bank_latency_failure = bank_latency.merge(
    bank_summary[['failure_rate', 'cancellation_rate', 'success_rate']],
    left_index=True,
    right_index=True,
    how='left'
)

latency_failure_corr = bank_latency_failure[['p95_latency', 'failure_rate']].corr().iloc[0, 1]

print(f"Correlation between bank p95 latency and failure rate: {latency_failure_corr:.3f}")

print(
    bank_latency_failure
    .head(15)[['bank_short','payments','median_latency','p95_latency',
               'failure_rate','cancellation_rate','success_rate']]
    .to_string()
)

#### Q5.3 Answer

Latency differs strongly by connectivity type. `type 2` and missing connectivity values have the highest p95 latency, while `type 8` has a much lower p95 latency. which means that most volume may be processed through a relatively faster connectivity type, while some of the smaller connectivity groups have long-tail latency.

At the bank level, the worst p95 latency values are extremely high. This means that some banks have a small but important group of very slow payments, even if their median latency is not always high.

The correlation between bank p95 latency and failure rate is 0.017, which is very close to zero. Based on this, high latency does not appear to be strongly related to high failure rate at the bank system


In [ ]:
# ============================================================
# Section 6 — User Classification
# ============================================================

# 6.1  Add user_stage column

df = df.sort_values(['user_id', 'createdat_ts']).copy()

df['attempt_number'] = df.groupby('user_id').cumcount() + 1

first_success = (
    df[df['is_executed']]
    .groupby('user_id')['createdat_ts']
    .min()
)

df['first_success_at'] = df['user_id'].map(first_success)

df['user_stage'] = 'New User'

df.loc[df['attempt_number'] == 1, 'user_stage'] = 'First Attempt'

df.loc[
    df['first_success_at'].notna() &
    (df['createdat_ts'] > df['first_success_at']),
    'user_stage'
] = 'Returning User'

print(df[['user_id','createdat_ts','status_clean','attempt_number',
          'first_success_at','user_stage']].head(20).to_string(index=False))

In [ ]:
# Check user_stage distribution

user_stage_counts = (
    df['user_stage']
    .value_counts()
    .reset_index()
)

user_stage_counts.columns = ['user_stage', 'payments']

print(user_stage_counts.to_string(index=False))

In [ ]:
# 6.2  Payment count, conversion rate, and average amount by user_stage

user_stage_summary = (
    df.groupby('user_stage')
      .agg(
          payments=('id', 'count'),
          executed=('is_executed', 'sum'),
          avg_amount=('amount_in_currency', 'mean')
      )
      .reset_index()
)

user_stage_summary['conversion_rate'] = (
    user_stage_summary['executed'] / user_stage_summary['payments'] * 100
).round(2)

user_stage_summary['avg_amount'] = user_stage_summary['avg_amount'].round(2)

user_stage_summary = user_stage_summary[
    ['user_stage', 'payments', 'executed', 'conversion_rate', 'avg_amount']
]

print(user_stage_summary.to_string(index=False))

In [ ]:
# Grouped bar chart for conversion rate and average amount

x = np.arange(len(user_stage_summary))
width = 0.35

fig, ax1 = plt.subplots(figsize=(9, 5))

bars1 = ax1.bar(
    x - width / 2,
    user_stage_summary['conversion_rate'],
    width,
    label='Conversion rate'
)

ax1.set_ylabel('Conversion Rate (%)')
ax1.set_xlabel('User Stage')
ax1.set_title('Conversion Rate and Average Amount by User Stage')
ax1.set_xticks(x)
ax1.set_xticklabels(user_stage_summary['user_stage'], rotation=20)

ax2 = ax1.twinx()

bars2 = ax2.bar(
    x + width / 2,
    user_stage_summary['avg_amount'],
    width,
    alpha=0.6,
    label='Average amount'
)

ax2.set_ylabel('Average Amount')

fig.legend(
    loc='upper right',
    bbox_to_anchor=(0.9, 0.9)
)

plt.tight_layout()
plt.show()

#### Q6.1–Q6.2 Answer

Users were classified using the homework definitions: first attempt is the first payment attempt ever, new user attempts are attempts up to and including the first successful payment, and returning user attempts are attempts after the first success.

Most payments are from returning users. There are 499,489 returning-user attempts, compared with only 173 first attempts and 338 new-user attempts. This shows that payment activity is heavily concentrated among users who have already succeeded at least once.

Returning users have the highest conversion rate at 75.57%. First attempts convert at 39.31%, while new-user attempts convert at 25.15%. Returning users likely convert better because they have already completed the payment journey before, may be using familiar banks or flows, and are less likely to abandon during setup or authorization.

In [ ]:
# 6.3  Days from first attempt to first successful payment

first_attempt = (
    df.groupby('user_id')['createdat_ts']
      .min()
)

first_success = (
    df[df['is_executed']]
    .groupby('user_id')['createdat_ts']
    .min()
)

user_conversion_time = pd.DataFrame({
    'first_attempt_at': first_attempt,
    'first_success_at': first_success
})

user_conversion_time['days_to_first_success'] = (
    user_conversion_time['first_success_at'] - user_conversion_time['first_attempt_at']
).dt.total_seconds() / (60 * 60 * 24)

user_conversion_time = user_conversion_time[
    user_conversion_time['days_to_first_success'].notna() &
    (user_conversion_time['days_to_first_success'] >= 0)
].copy()

median_days = user_conversion_time['days_to_first_success'].median()
converted_7d_pct = (
    (user_conversion_time['days_to_first_success'] <= 7).mean() * 100
)

print(f"Users with first success: {len(user_conversion_time):,}")
print(f"Median days to first success: {median_days:.2f}")
print(f"% converting within 7 days: {converted_7d_pct:.2f}%")

print(user_conversion_time.head(10).to_string())

In [ ]:
# Plot distribution capped at 60 days

days_capped = user_conversion_time[
    user_conversion_time['days_to_first_success'] <= 60
]['days_to_first_success']

fig, ax = plt.subplots(figsize=(10, 5))

ax.hist(
    days_capped,
    bins=30,
    alpha=0.75
)

ax.axvline(median_days, linestyle='--', linewidth=2, label='Median')

ax.set_title('Days from First Attempt to First Successful Payment')
ax.set_xlabel('Days to First Success')
ax.set_ylabel('User Count')
ax.legend()

plt.tight_layout()
plt.show()

#### Q6.3 Answer

Out of 173 users, 153 eventually make a first successful payment. The median time from first attempt to first success is 1.09 days, which means that users who convert usually do so quickly.

62.75% of users with a first success convert within 7 days. The distribution is right-skewed: most successful users convert early, but a smaller group takes several weeks before their first successful payment.

This suggests that the first few days after a user’s first attempt are especially important for activation. Users who do not succeed quickly may need better retry flows, clearer error messages, or follow-up support.

In [ ]:
# ============================================================
# Section 7 — Frequency Tiers & RFM Analysis
# ============================================================

# 7.1  Frequency tiers using executed payments only

user_freq = (
    executed_df
    .groupby('user_id')
    .agg(
        executed_payments=('id', 'count'),
        total_tpv=('amount_in_currency', 'sum')
    )
)

def assign_frequency_tier(x):
    if x >= 15:
        return 'Champion'
    elif x >= 8:
        return 'Super'
    elif x >= 4:
        return 'Engaged'
    elif x >= 2:
        return 'Light'
    else:
        return 'Inactive'

user_freq['frequency_tier'] = user_freq['executed_payments'].apply(assign_frequency_tier)

tier_summary = (
    user_freq
    .groupby('frequency_tier')
    .agg(
        users=('executed_payments', 'count'),
        total_tpv=('total_tpv', 'sum'),
        avg_executed_payments=('executed_payments', 'mean')
    )
    .reset_index()
)

tier_summary['user_pct'] = (
    tier_summary['users'] / tier_summary['users'].sum() * 100
).round(2)

tier_summary['tpv_pct'] = (
    tier_summary['total_tpv'] / tier_summary['total_tpv'].sum() * 100
).round(2)

tier_summary['avg_executed_payments'] = tier_summary['avg_executed_payments'].round(2)
tier_summary['total_tpv'] = tier_summary['total_tpv'].round(2)

tier_order = ['Champion', 'Super', 'Engaged', 'Light', 'Inactive']
tier_summary['tier_order'] = tier_summary['frequency_tier'].map({
    'Champion': 1,
    'Super': 2,
    'Engaged': 3,
    'Light': 4,
    'Inactive': 5
})

tier_summary = tier_summary.sort_values('tier_order').drop(columns='tier_order')

print(tier_summary.to_string(index=False))

In [ ]:
# Pareto table: cumulative users vs cumulative TPV

pareto = user_freq.sort_values('total_tpv', ascending=False).copy()

pareto['user_rank'] = np.arange(1, len(pareto) + 1)

pareto['cum_users_pct'] = (
    pareto['user_rank'] / len(pareto) * 100
).round(2)

pareto['cum_tpv_pct'] = (
    pareto['total_tpv'].cumsum() / pareto['total_tpv'].sum() * 100
).round(2)

top_20_cutoff = int(np.ceil(len(pareto) * 0.20))
top_20_tpv_pct = pareto.iloc[top_20_cutoff - 1]['cum_tpv_pct']

print(f"Users in Pareto table: {len(pareto):,}")
print(f"Top 20% users TPV share: {top_20_tpv_pct:.2f}%")

print(pareto.head(15)[
    ['user_rank', 'executed_payments', 'total_tpv', 'frequency_tier',
     'cum_users_pct', 'cum_tpv_pct']
].to_string(index=False))

In [ ]:
# Pareto curve

fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(
    pareto['cum_users_pct'],
    pareto['cum_tpv_pct'],
    linewidth=2
)

ax.axvline(20, linestyle='--', linewidth=1)
ax.axhline(80, linestyle='--', linewidth=1)

ax.set_title('Pareto Curve: Users vs TPV')
ax.set_xlabel('Cumulative Users (%)')
ax.set_ylabel('Cumulative TPV (%)')

plt.tight_layout()
plt.show()

Top 20% users drive 97.51% of TPV, so the answer to the 80/20 question is yes. Just a small group of users drive almost all of the payments, so retaining this high frequency users is mandatory.

In [ ]:
# 7.2  RFM scoring

SNAPSHOT_DATE = df['createdat_ts'].max()

rfm = (
    df[df['is_executed']]
    .groupby('user_id')
    .agg(
        last_payment=('createdat_ts','max'),
        frequency=('id','count'),
        monetary=('amount_in_currency','sum')
    )
    .reset_index()
)

rfm['recency_days'] = (SNAPSHOT_DATE - rfm['last_payment']).dt.days

print('RFM base stats:')
print(rfm[['recency_days','frequency','monetary']].describe().round(1))

In [ ]:
# Score each dimension 1–5 using quintiles

def quintile_score(series, ascending=True):
    labels = [5,4,3,2,1] if ascending else [1,2,3,4,5]
    return pd.qcut(
        series.rank(method='first'),
        q=5,
        labels=labels
    ).astype(int)

rfm['R'] = quintile_score(rfm['recency_days'], ascending=True)   # lower recency → better → score 5
rfm['F'] = quintile_score(rfm['frequency'], ascending=False)
rfm['M'] = quintile_score(rfm['monetary'], ascending=False)

rfm['RFM_score'] = rfm['R'].astype(str) + rfm['F'].astype(str) + rfm['M'].astype(str)
rfm['RFM_total'] = rfm['R'] + rfm['F'] + rfm['M']

def rfm_segment(row):
    r, f, m = row['R'], row['F'], row['M']
    if r >= 4 and f >= 4 and m >= 4: return 'Champions'
    if r >= 3 and f >= 3:            return 'Loyal'
    if r >= 4 and f <= 2:            return 'Promising'
    if r <= 2 and f >= 3:            return 'At Risk'
    if r <= 2 and f <= 2 and m >= 3: return 'Needs Attention'
    if r == 1 and f == 1:            return 'Lost'
    return 'Hibernating'

rfm['segment'] = rfm.apply(rfm_segment, axis=1)

print(rfm['segment'].value_counts())

In [ ]:
# Segment summary table

seg_summary = (
    rfm.groupby('segment')
    .agg(
        users=('user_id','count'),
        avg_recency=('recency_days','mean'),
        avg_frequency=('frequency','mean'),
        avg_monetary=('monetary','mean'),
        total_tpv=('monetary','sum')
    )
    .round(1)
    .sort_values('total_tpv', ascending=False)
)

print(seg_summary.to_string())

#### Q7.2 Answer

Recency is based on days since the user’s last executed payment, frequency is the number of executed payments, and monetary value is total executed TPV.

The strongest segment `Champions': 40 users generate 9.68 million TPV, being the largest segment by value and these users are very recent, very frequent, and high value.

The `At Risk` segment has 21 users and 437.6k TPV, but average recency is 437.9 days, meaning these users used to be valuable but have not paid recently.

The `Lost` and `Needs Attention` segments have much lower total TPV, but they show users who are either inactive or slipping away. The main business priority should be retaining Champions and trying to reactivate At Risk users.

In [ ]:
# 7.3  Bubble chart by RFM segment

seg_plot = seg_summary.reset_index().copy()

fig, ax = plt.subplots(figsize=(11, 6))

ax.scatter(
    seg_plot['avg_recency'],
    seg_plot['avg_frequency'],
    s=seg_plot['total_tpv'] / seg_plot['total_tpv'].max() * 2500,
    alpha=1
)

for _, row in seg_plot.iterrows():
    ax.text(
        row['avg_recency'],
        row['avg_frequency'],
        row['segment'],
        fontsize=9
    )

ax.set_title('RFM Segments: Recency vs Frequency')
ax.set_xlabel('Average Recency Days')
ax.set_ylabel('Average Frequency')

plt.tight_layout()
plt.show()

In [ ]:
# 7.4  RFM segment by vertical

user_vertical = (
    df[df['is_executed']]
    .groupby(['user_id', 'vertical'])
    .agg(payments=('id','count'))
    .reset_index()
)

user_vertical = (
    user_vertical
    .sort_values(['user_id', 'payments'], ascending=[True, False])
    .drop_duplicates('user_id')
)

rfm_vertical = rfm.merge(
    user_vertical[['user_id','vertical']],
    on='user_id',
    how='left'
)

segment_vertical = pd.crosstab(
    rfm_vertical['segment'],
    rfm_vertical['vertical'],
    normalize='index'
).round(3)

print(segment_vertical.to_string())

In [ ]:
# Heatmap-style plot using matplotlib

fig, ax = plt.subplots(figsize=(8, 5))

im = ax.imshow(segment_vertical, aspect='auto')

ax.set_title('RFM Segment by Vertical')
ax.set_xlabel('Vertical')
ax.set_ylabel('RFM Segment')

ax.set_xticks(range(len(segment_vertical.columns)))
ax.set_xticklabels(segment_vertical.columns, rotation=30)

ax.set_yticks(range(len(segment_vertical.index)))
ax.set_yticklabels(segment_vertical.index)

cbar = plt.colorbar(im, ax=ax)
cbar.set_label('Row-normalized User Share')

plt.tight_layout()
plt.show()

#### Q7.3–Q7.4 Answer

The RFM bubble chart shows that the `Champions` segment is very different from the rest of the user base. Champions have very low average recency, extremely high average frequency, and the largest total TPV. This is why their bubble dominates the chart.

Other segments sit much lower on the frequency axis. `At Risk` users still have meaningful total TPV, but their average recency is much higher, meaning they have not been active recently. `Lost`, `Hibernating`, and `Needs Attention` users have lower frequency and lower total value.

The vertical heatmap is row-normalized, so each row shows how users within that RFM segment are distributed across verticals. Vertical 2 is the largest vertical across most segments. For example, 55% of Champions and 58% of Loyal users are mainly associated with vertical 2. Champions are also strongly represented in vertical 5, with 38% of Champion users assigned to that vertical.

Overall, the main business message is that high-value users are concentrated in a small number of RFM segments and mostly in vertical 2 and vertical 5. Retention work should focus first on Champions and At Risk users, especially within these major verticals.

In [ ]:
# ============================================================
# Section 8 — Retention & Growth Accounting
# ============================================================

# 8.1  Build event-level 90-day forward retention table

from datetime import timedelta

SNAPSHOT    = df['createdat_ts'].max()
WINDOW_DAYS = 90

events = df[df['is_executed']][['user_id','createdat_ts','created_month','vertical']].copy()
events = events.rename(columns={'createdat_ts':'payment_date'})
events = events.sort_values(['user_id','payment_date'])

events['next_payment'] = events.groupby('user_id')['payment_date'].shift(-1)
events['window_end']   = events['payment_date'] + timedelta(days=WINDOW_DAYS)
events['censored']     = events['window_end'] > SNAPSHOT

events['retained'] = (
    events['next_payment'].notna() &
    (events['next_payment'] <= events['window_end'])
)

eligible = events[~events['censored']]
rate = eligible['retained'].sum() / len(eligible) * 100

print(f'Total events       : {len(events):,}')
print(f'Censored           : {events["censored"].sum():,}')
print(f'Eligible           : {len(eligible):,}')
print(f'Retained           : {int(eligible["retained"].sum()):,}')
print(f'90-Day Retention   : {rate:.2f}%')

In [ ]:
# 8.1  Monthly retention rate trend

events['month'] = events['payment_date'].dt.to_period('M')

monthly_ret = (
    events[~events['censored']]
    .groupby('month')
    .agg(
        eligible=('user_id','count'),
        retained=('retained','sum')
    )
    .reset_index()
)

monthly_ret['retention_rate'] = (
    monthly_ret['retained'] / monthly_ret['eligible'] * 100
).round(2)

monthly_ret['month_str'] = monthly_ret['month'].astype(str)

print(monthly_ret.head().to_string(index=False))
print()
print(monthly_ret.tail().to_string(index=False))

In [ ]:
# 8.1  Plot monthly 90-day retention rate

fig, ax = plt.subplots(figsize=(14, 4))

ax.plot(
    monthly_ret['month_str'],
    monthly_ret['retention_rate'],
    marker='o',
    linewidth=2
)

ax.axhline(
    monthly_ret['retention_rate'].mean(),
    linestyle='--',
    label=f"Avg: {monthly_ret['retention_rate'].mean():.2f}%"
)

ax.set_title('Monthly 90-Day Retention Rate')
ax.set_xlabel('Month')
ax.set_ylabel('Retention %')
ax.set_ylim(95, 100.2)
ax.legend()

plt.xticks(rotation=45, ha='right', fontsize=7)
plt.tight_layout()
plt.show()

In [ ]:
# 8.1  90-day retention by vertical

vertical_ret = (
    events[~events['censored']]
    .groupby('vertical', dropna=False)
    .agg(
        eligible=('user_id','count'),
        retained=('retained','sum')
    )
    .reset_index()
)

vertical_ret['retention_rate'] = (
    vertical_ret['retained'] / vertical_ret['eligible'] * 100
).round(2)

vertical_ret = vertical_ret.sort_values('retention_rate', ascending=False)

print(vertical_ret.to_string(index=False))

#### Q8.1 Answer

The 90-day forward retention rate is very high. Out of 299,576 eligible executed payment events, 299,446 had another executed payment within the next 90 days, giving a 90-day retention rate of 99.96%.

The monthly trend is also stable. Most months are close to 100%, so the chart uses a zoomed y-axis to make the small differences easier to see. Without zooming, the line would look almost completely flat because the values are all near the top of the 0–100% range.

Retention is also high across the main verticals. Vertical 5, vertical 2, and vertical 4 are all above 99%, while vertical 3 and vertical 1 are lower but still high. The row with missing vertical should not be interpreted strongly because it has only 2 eligible payments.

In [ ]:
# 8.2  Build cohort retention table

user_months = (
    df[df['is_executed']]
    .assign(month=lambda x: x['createdat_ts'].dt.to_period('M'))
    .groupby(['user_id','month'])
    .size()
    .reset_index(name='payments')
)

first_month = user_months.groupby('user_id')['month'].min().rename('cohort_month')
user_months = user_months.join(first_month, on='user_id')

user_months['period_number'] = (
    (user_months['month'].dt.year - user_months['cohort_month'].dt.year) * 12 +
    (user_months['month'].dt.month - user_months['cohort_month'].dt.month)
)

cohort_counts = (
    user_months[user_months['period_number'].between(0, 12)]
    .groupby(['cohort_month','period_number'])
    .agg(users=('user_id','nunique'))
    .reset_index()
)

cohort_sizes = (
    cohort_counts[cohort_counts['period_number'] == 0]
    [['cohort_month','users']]
    .rename(columns={'users':'cohort_size'})
)

cohort_counts = cohort_counts.merge(
    cohort_sizes,
    on='cohort_month',
    how='left'
)

cohort_counts['retention_rate'] = (
    cohort_counts['users'] / cohort_counts['cohort_size'] * 100
).round(2)

print(cohort_counts.head(20).to_string(index=False))

In [ ]:
# 8.2  Cohort retention matrix, months +1 to +12

cohort_matrix = cohort_counts.pivot_table(
    index='cohort_month',
    columns='period_number',
    values='retention_rate'
)

cohort_matrix_12 = cohort_matrix.loc[:, cohort_matrix.columns.isin(range(1, 13))].copy()

print(cohort_matrix_12.head(20).to_string())

In [ ]:
# 8.2  Plot cohort retention heatmap

fig, ax = plt.subplots(figsize=(12, 9))

cohort_plot = np.ma.masked_invalid(cohort_matrix_12.values)

im = ax.imshow(
    cohort_plot,
    aspect='auto'
)

ax.set_title('Cohort Retention Matrix: Months +1 to +12')
ax.set_xlabel('Months Since First Executed Payment')
ax.set_ylabel('Cohort Month')

ax.set_xticks(range(len(cohort_matrix_12.columns)))
ax.set_xticklabels(cohort_matrix_12.columns)

ax.set_yticks(range(len(cohort_matrix_12.index)))
ax.set_yticklabels(cohort_matrix_12.index.astype(str))

cbar = plt.colorbar(im, ax=ax)
cbar.set_label('Retention %')

plt.tight_layout()
plt.show()

In [ ]:
# 8.2  Best 3-month retention cohort

month_3 = (
    cohort_counts[cohort_counts['period_number'] == 3]
    .sort_values(['retention_rate','cohort_size'], ascending=[False, False])
    .copy()
)

print('Top cohorts by month-3 retention:')
print(
    month_3[
        ['cohort_month','users','cohort_size','retention_rate']
    ]
    .head(10)
    .to_string(index=False)
)

print()
print('Best 3-month retention cohort:')
print(
    month_3[
        ['cohort_month','retention_rate','cohort_size']
    ]
    .head(1)
    .to_string(index=False)
)

#### Q8.2 Answer

The cohort retention matrix tracks users from the month of their first executed payment and follows whether they remain active in later months. Each row is a cohort and columns show the number of months since that cohort's first successful payment.

The matrix shows us an overall high retention with many cohorts reaching 100% in several periods. but we do need to interpret carefully because many cohorts are very small. In small cohorts, one user can move the retention rate a lot, so a 100% value does not always mean that its a strong patternn.

The main takeaway is that retention looks strong overall, but the cohort matrix is better used as supporting evidence than the main business conclusion.

In [ ]:
# 8.3  Build monthly growth accounting table

user_months = (
    df[df['is_executed']]
    .assign(month=lambda x: x['createdat_ts'].dt.to_period('M'))
    .groupby(['user_id','month'])
    .size()
    .reset_index(name='payments')
)

first_month = user_months.groupby('user_id')['month'].min().rename('first_month')
user_months = user_months.join(first_month, on='user_id')

all_months = sorted(user_months['month'].unique())
result = []

for i, month in enumerate(all_months):
    curr_users = set(user_months[user_months['month'] == month]['user_id'])
    prev_users = set(user_months[user_months['month'] == all_months[i-1]]['user_id']) if i > 0 else set()

    new_users   = {u for u in curr_users if first_month[u] == month}
    retained    = curr_users & prev_users - new_users
    resurrected = curr_users - prev_users - new_users
    churned     = prev_users - curr_users

    result.append({
        'month': str(month),
        'MAU': len(curr_users),
        'new': len(new_users),
        'retained': len(retained),
        'resurrected': len(resurrected),
        'churned': len(churned),
        'net_change': len(new_users) + len(resurrected) - len(churned)
    })

growth = pd.DataFrame(result)

print(growth.to_string(index=False))

In [ ]:
# 8.3  Growth accounting waterfall chart

fig, ax = plt.subplots(figsize=(14, 5))

ax.bar(
    growth['month'],
    growth['new'],
    label='New'
)

ax.bar(
    growth['month'],
    growth['resurrected'],
    bottom=growth['new'],
    label='Resurrected'
)

ax.bar(
    growth['month'],
    -growth['churned'],
    label='Churned'
)

ax.axhline(0, linewidth=0.8)

ax.set_title('Monthly Growth Accounting: New + Resurrected - Churned')
ax.set_xlabel('Month')
ax.set_ylabel('Users')
ax.legend()

plt.xticks(rotation=45, ha='right', fontsize=7)
plt.tight_layout()
plt.show()

In [ ]:
# 8.3  Last 6 months growth check

last_6_growth = growth.tail(6).copy()

print('Last 6 months:')
print(last_6_growth.to_string(index=False))

print()
print(f'Net change in last 6 months: {last_6_growth["net_change"].sum():,.0f}')

if last_6_growth['net_change'].sum() > 0:
    print('User base is growing in the last 6 months.')
elif last_6_growth['net_change'].sum() < 0:
    print('User base is contracting in the last 6 months.')
else:
    print('User base is flat in the last 6 months.')

#### Q8.3 Answer

The growth accounting table breaks monthly active users into new, retained, resurrected, and churned users. This makes the MAU trend easier to understand because we can see whether growth is coming from new acquisition, returning users, or existing users staying active.

The main driver of the user base is retained users. By the end of the period, retained users make up most monthly active users: for example, 2010-01 had 82 MAU, with 72 retained users. New users are much smaller, usually only a few users per month, while resurrected users add some extra activity but do not drive the trend on their own.

The last 6 months show a net change of -7 users. Most of this comes from 2010-02, where MAU dropped from 82 to 66 and churn increased to 18 users. This suggests a recent slowdown or contraction after a long period of growth. However, 2010-02 should be interpreted carefully because it may be a partial month at the end of the dataset.

Overall, the user base looks sticky because retention is strong, but growth is not mainly coming from new users. The business depends heavily on keeping existing active users engaged, and the latest months show that churn can quickly offset new and resurrected users.